In [4]:
import re
import numpy as np
import xarray as xr
import rioxarray
from collections import defaultdict
from pathlib import Path


def build_union_grid(rasters, res=None):
    """Compute union extent and create a target grid."""
    bounds = []
    resolutions = []
    for f in rasters:
        da = rioxarray.open_rasterio(f).squeeze()
        bounds.append(da.rio.bounds())
        resolutions.append(da.rio.resolution())
    minx = min(b[0] for b in bounds)
    miny = min(b[1] for b in bounds)
    maxx = max(b[2] for b in bounds)
    maxy = max(b[3] for b in bounds)

    # Pick resolution (finest among all rasters if not specified)
    if res is None:
        resx = min(abs(r[0]) for r in resolutions)
        resy = min(abs(r[1]) for r in resolutions)
    else:
        resx, resy = res

    xs = np.arange(minx, maxx + resx, resx)
    ys = np.arange(maxy, miny - resy, -resy)  # descending
    target = xr.DataArray(
        np.empty((len(ys), len(xs)), dtype="float32"),
        coords={"y": ys, "x": xs},
        dims=("y", "x"),
    )
    target.rio.write_crs("EPSG:4326", inplace=True)
    return target


# --- Setup paths ---
folder = Path("D:/MyDrive/Stability/RawData/Monthly_Averages/Beaufort_month_year")
out_dir = folder / "WeightedMeans_Union"
out_dir.mkdir(exist_ok=True)

# Match: Region_Monthly_{Count|Mean}_YYYY_MM.tif
pat = re.compile(r"^(.*?)_Monthly_(Count|Mean)_(\d{4})_(\d{2})\.tif$")

# Group by (year, month)
by_year_month = defaultdict(lambda: {"Count": [], "Mean": []})

for f in folder.glob("*.tif"):
    m = pat.match(f.name)
    if m:
        region, ftype, year, month = m.groups()
        key = (int(year), int(month))
        by_year_month[key][ftype].append(f)

# --- Process each year-month group ---
for (year, month), groups in sorted(by_year_month.items()):
    files = groups["Count"] + groups["Mean"]
    if not files:
        continue

    print(f"\nProcessing {year}-{month:02d} with {len(groups['Count'])} regions")

    # Build union grid
    target_grid = build_union_grid(files)

    weighted_sum = None
    total_count = None

    for cfile, mfile in zip(sorted(groups["Count"]), sorted(groups["Mean"])):
        count = (
            rioxarray.open_rasterio(cfile, masked=True, chunks={"x": 1024, "y": 1024})
            .squeeze()
            .astype("float32")
        )
        mean = (
            rioxarray.open_rasterio(mfile, masked=True, chunks={"x": 1024, "y": 1024})
            .squeeze()
            .astype("float32")
        )

        # Mask out pixels where count < 1
        mask = count < 1
        count = count.where(~mask)
        mean = mean.where(~mask)

        # Skip if all NaN
        if not count.notnull().any():
            print(f"  Skipping {cfile} (all nodata after masking)")
            continue

        # Reproject
        count = count.rio.reproject_match(target_grid)
        mean = mean.rio.reproject_match(target_grid)

        # Weighted contribution
        weighted = mean * count

        # Incremental accumulation (preserves mean where count == 1)
        if weighted_sum is None:
            weighted_sum = weighted
            total_count = count
        else:
            weighted_sum = weighted_sum.fillna(0) + weighted.fillna(0)
            total_count = total_count.fillna(0) + count.fillna(0)

    if weighted_sum is None:
        print(f"No valid rasters for {year}-{month:02d}, skipping")
        continue

    # Final weighted mean
    weighted_mean = weighted_sum / total_count
    weighted_mean = weighted_mean.where(total_count >= 1)

    out_file = out_dir / f"WeightedMean_{year}_{month:02d}.tif"
    weighted_mean.rio.to_raster(out_file, compress="deflate")
    print(f"Saved {out_file}")



Processing 2017-01 with 1 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\WeightedMeans_Union\WeightedMean_2017_01.tif

Processing 2017-02 with 6 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\WeightedMeans_Union\WeightedMean_2017_02.tif

Processing 2017-03 with 6 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\WeightedMeans_Union\WeightedMean_2017_03.tif

Processing 2017-04 with 3 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\WeightedMeans_Union\WeightedMean_2017_04.tif

Processing 2017-05 with 6 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\WeightedMeans_Union\WeightedMean_2017_05.tif

Processing 2017-06 with 6 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\WeightedMeans_Union\WeightedMean_2017_06.tif

Processing 2017-07 with 4 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\B

In [1]:
import re
import numpy as np
import xarray as xr
import rioxarray
from collections import defaultdict
from pathlib import Path


def build_union_grid(rasters, res=None):
    """Compute union extent and create a target grid."""
    bounds = []
    resolutions = []
    for f in rasters:
        da = rioxarray.open_rasterio(f).squeeze()
        bounds.append(da.rio.bounds())
        resolutions.append(da.rio.resolution())
    minx = min(b[0] for b in bounds)
    miny = min(b[1] for b in bounds)
    maxx = max(b[2] for b in bounds)
    maxy = max(b[3] for b in bounds)

    # Pick resolution (finest among all rasters if not specified)
    if res is None:
        resx = min(abs(r[0]) for r in resolutions)
        resy = min(abs(r[1]) for r in resolutions)
    else:
        resx, resy = res

    xs = np.arange(minx, maxx + resx, resx)
    ys = np.arange(maxy, miny - resy, -resy)  # descending
    target = xr.DataArray(
        np.empty((len(ys), len(xs)), dtype="float32"),
        coords={"y": ys, "x": xs},
        dims=("y", "x"),
    )
    target.rio.write_crs("EPSG:4326", inplace=True)
    return target


# --- Setup paths ---
folder = Path("D:/MyDrive/Stability/RawData/Monthly_Averages/Chukchi_month_year")
out_dir = folder / "WeightedMeans_Union"
out_dir.mkdir(exist_ok=True)

# Match: Region_Monthly_{Count|Mean}_YYYY_MM.tif
pat = re.compile(r"^(.*?)_Monthly_(Count|Mean)_(\d{4})_(\d{2})\.tif$")

# Group by (year, month)
by_year_month = defaultdict(lambda: {"Count": [], "Mean": []})

for f in folder.glob("*.tif"):
    m = pat.match(f.name)
    if m:
        region, ftype, year, month = m.groups()
        key = (int(year), int(month))
        by_year_month[key][ftype].append(f)

# --- Process each year-month group ---
for (year, month), groups in sorted(by_year_month.items()):
    files = groups["Count"] + groups["Mean"]
    if not files:
        continue

    print(f"\nProcessing {year}-{month:02d} with {len(groups['Count'])} regions")

    # Build union grid
    target_grid = build_union_grid(files)

    weighted_sum = None
    total_count = None

    for cfile, mfile in zip(sorted(groups["Count"]), sorted(groups["Mean"])):
        count = (
            rioxarray.open_rasterio(cfile, masked=True, chunks={"x": 1024, "y": 1024})
            .squeeze()
            .astype("float32")
        )
        mean = (
            rioxarray.open_rasterio(mfile, masked=True, chunks={"x": 1024, "y": 1024})
            .squeeze()
            .astype("float32")
        )

        # Mask out pixels where count < 1
        mask = count < 1
        count = count.where(~mask)
        mean = mean.where(~mask)

        # Skip if all NaN
        if not count.notnull().any():
            print(f"  Skipping {cfile} (all nodata after masking)")
            continue

        # Reproject
        count = count.rio.reproject_match(target_grid)
        mean = mean.rio.reproject_match(target_grid)

        # Weighted contribution
        weighted = mean * count

        # Incremental accumulation (preserves mean where count == 1)
        if weighted_sum is None:
            weighted_sum = weighted
            total_count = count
        else:
            weighted_sum = weighted_sum.fillna(0) + weighted.fillna(0)
            total_count = total_count.fillna(0) + count.fillna(0)

    if weighted_sum is None:
        print(f"No valid rasters for {year}-{month:02d}, skipping")
        continue

    # Final weighted mean
    weighted_mean = weighted_sum / total_count
    weighted_mean = weighted_mean.where(total_count >= 1)

    out_file = out_dir / f"WeightedMean_{year}_{month:02d}.tif"
    weighted_mean.rio.to_raster(out_file, compress="deflate")
    print(f"Saved {out_file}")


Processing 2016-07 with 1 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi_month_year\WeightedMeans_Union\WeightedMean_2016_07.tif

Processing 2016-08 with 1 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi_month_year\WeightedMeans_Union\WeightedMean_2016_08.tif

Processing 2016-09 with 1 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi_month_year\WeightedMeans_Union\WeightedMean_2016_09.tif

Processing 2016-10 with 1 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi_month_year\WeightedMeans_Union\WeightedMean_2016_10.tif

Processing 2016-11 with 1 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi_month_year\WeightedMeans_Union\WeightedMean_2016_11.tif

Processing 2016-12 with 1 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi_month_year\WeightedMeans_Union\WeightedMean_2016_12.tif

Processing 2017-01 with 1 regions
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi